# 08 - Cheques de Pago Diferido (CPD)

Este notebook prueba las funcionalidades de CPD de la API de IOL.

## Funcionalidades:
- Verificar si se puede operar CPD
- Listar cheques disponibles
- Calcular comisiones
- Comprar cheques

**ADVERTENCIA:** Las operaciones de compra estan COMENTADAS.

**Nota:** Requiere credenciales validas de IOL en el archivo `.env`

## Configuracion Inicial

In [ ]:
import sys
import os
from datetime import datetime
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../..'))

from pyIol import (
    IOLClient, IOLAPIError,
    PuedeOperarCPD, ChequeCPD, ComisionesCPD, ResultadoCPD,
    CPDStates, CPDSegments
)
print("Librerias importadas correctamente")
print(f"Fecha y hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
load_dotenv('../../.env')
USERNAME = os.getenv('IOL_USERNAME', 'tu_usuario_iol')
PASSWORD = os.getenv('IOL_PASSWORD', 'tu_password_iol')

if USERNAME == "tu_usuario_iol":
    print("ADVERTENCIA: Configura las credenciales en .env")
else:
    print(f"Credenciales configuradas - Usuario: {USERNAME}")

In [ ]:
try:
    client = IOLClient(USERNAME, PASSWORD)
    print("Cliente IOL creado correctamente")
except Exception as e:
    print(f"Error al crear cliente: {e}")
    client = None

## 1. Verificar si se puede Operar CPD

In [ ]:
# Verificar habilitacion
if client:
    try:
        print("Verificando habilitacion para operar CPD...")
        puede = client.can_operate_cpd()
        
        print(f"\nPuede operar CPD: {'Si' if puede.puede_operar else 'No'}")
        if puede.mensaje:
            print(f"Mensaje: {puede.mensaje}")
    except Exception as e:
        print(f"Error: {e}")

## 2. Listar Cheques Disponibles

In [ ]:
# Listar cheques
if client:
    try:
        print("Obteniendo lista de cheques disponibles...")
        cheques = client.get_cpd_list(
            estado=CPDStates.DISPONIBLES,
            segmento=CPDSegments.AVALADOS
        )
        
        if cheques:
            print(f"\nCheques encontrados: {len(cheques)}")
            
            print("\nPrimeros 5 cheques:")
            for i, cheque in enumerate(cheques[:5], 1):
                print(f"\n  {i}. Cheque #{cheque.numero_cheque}")
                print(f"     Importe: ${cheque.importe:,.2f}")
                print(f"     Valor presente: ${cheque.valor_presente:,.2f}")
                print(f"     Vencimiento: {cheque.fecha_vencimiento}")
                print(f"     Plazo: {cheque.plazo} dias")
                print(f"     Tasa: {cheque.tasa:.2f}%")
        else:
            print("No hay cheques disponibles")
    except Exception as e:
        print(f"Error: {e}")

## 3. Calcular Comisiones

In [ ]:
# Calcular comisiones
if client:
    try:
        print("Calculando comisiones para un cheque de $100,000 a 90 dias...")
        comisiones = client.get_cpd_commissions(
            importe=100000.0,
            plazo=90,
            tasa=45.0
        )
        
        if comisiones:
            print(f"\nDesglose de comisiones:")
            print(f"  Comision IOL: ${comisiones.comision_iol:,.2f}")
            print(f"  Derechos de mercado: ${comisiones.derechos_mercado:,.2f}")
            print(f"  IVA: ${comisiones.iva:,.2f}")
            print(f"  Total gastos: ${comisiones.total_gastos:,.2f}")
    except Exception as e:
        print(f"Error: {e}")

## 4. Comprar Cheque (COMENTADO)

In [ ]:
# COMPRA DE CHEQUE - COMENTADO POR SEGURIDAD

# if client:
#     try:
#         # Primero obtener un cheque disponible
#         cheques = client.get_cpd_list(estado=CPDStates.DISPONIBLES)
#         
#         if cheques:
#             cheque = cheques[0]
#             
#             resultado = client.operate_cpd(
#                 numero_cheque=cheque.numero_cheque,
#                 precio=cheque.valor_presente
#             )
#             
#             if resultado.ok:
#                 print(f"Compra exitosa!")
#                 print(f"  Operacion: #{resultado.numero_operacion}")
#             else:
#                 print(f"Error: {resultado.mensaje}")
#     except Exception as e:
#         print(f"Error: {e}")

print("Compra de cheque COMENTADA - Descomenta para ejecutar")

## Metodos RAW Disponibles

In [ ]:
print("METODOS RAW DE CPD")
print("="*50)
print("""
Para obtener respuestas en formato JSON crudo:

- client.can_operate_cpd_raw()
- client.get_cpd_list_raw(estado, segmento)
- client.get_cpd_commissions_raw(importe, plazo, tasa)
- client.operate_cpd_raw(numero_cheque, precio, cantidad)

Estos metodos retornan el JSON exacto de la API de IOL.
""")

## Limpieza

In [ ]:
if client:
    try:
        client.close()
        print("Cliente IOL cerrado correctamente")
    except Exception as e:
        print(f"Error al cerrar cliente: {e}")